<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_09_decorators/note_lesson_09_decorators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 9 — Декоратори

> **Сценарій:** пишемо простий блог із трьома ролями користувачів — `guest`, `user`, `admin`. Кожна дія (переглянути пост, створити, редагувати, опублікувати, видалити, архівувати) повинна перевіряти, чи має користувач право її виконати. Почнемо з найпростішого рішення — і побачимо, чому воно не витримує реального життя.


## RETRIEVE

Коротке пригадування без підглядання (Урок 7):

- Як оголосити функцію з параметром і повернути значення через `return`?
- Що станеться, якщо функцію викликати без обов'язкового аргументу?
- `print(len)` — це валидний Python-код? Що він виведе?

Остання підказка важлива: ім'я функції без дужок — це те саме, що ім'я змінної. `len` тут — **значення**, яке можна надрукувати, передати далі, покласти в список. Дужки `()` — це те, що **викликає** функцію; без них ми просто тримаємо її як об'єкт. Саме на цьому спостереженні побудований увесь сьогоднішній урок.

## CONCEPT — функції це теж об'єкти

Перш ніж говорити про декоратори, перевіримо три речі про функції, які здаються дивними, поки не побачиш їх на власні очі.

In [1]:
def say_hello():
    print("Привіт!")


def run_function(func):
    print("--- Запускаємо функцію ---")
    func()  # викликаємо те, що передали
    print("--- Готово ---")


run_function(say_hello)  # передаємо say_hello БЕЗ дужок — саму функцію, а не результат її виклику

--- Запускаємо функцію ---
Привіт!
--- Готово ---


`run_function` отримала `say_hello` як звичайний аргумент — так само, як могла б отримати число чи рядок. Функція викликається лише там, де стоїть `func()`, тобто всередині `run_function`. Це і є перша властивість: **функцію можна передати як аргумент**.

In [2]:
def create_greeter(name):
    def greet():
        print(f"Привіт, {name}!")
    return greet  # повертаємо функцію, а НЕ результат її виклику (без дужок!)


greet_ivan = create_greeter("Іван")
greet_olia = create_greeter("Оля")

print(type(greet_ivan))
greet_ivan()
greet_olia()

<class 'function'>
Привіт, Іван!
Привіт, Оля!


`create_greeter` не друкує нічого сама — вона **будує й повертає нову функцію** `greet`, яка "пам'ятає" своє `name`. `greet_ivan` і `greet_olia` — це дві різні функції з різною "пам'яттю". Це друга властивість: **функція може повернути іншу функцію**. Те, що внутрішня функція запам'ятовує змінну із зовнішньої, називається **замиканням (closure)** — повернемось до цього за хвилину.

### Місток до декоратора

Об'єднаємо обидві властивості:

```text
функція → значення               (можна покласти в змінну, надрукувати)
        → можна передати далі     (як аргумент іншій функції)
        → можна повернути         (з іншої функції)
        → функція, що приймає функцію і повертає функцію
        → це і є декоратор
```

Декоратор — не нова синтаксична конструкція Python, а **застосування цих двох властивостей одночасно**: функція, яка приймає іншу функцію й повертає нову, "покращену" версію.

In [3]:
def make_counter():
    count = 0          # ця змінна живе в замиканні

    def increment():
        nonlocal count  # без цього рядка Python створив би НОВУ локальну count і впав би нижче
        count += 1
        return count

    return increment


counter_a = make_counter()
counter_b = make_counter()

print(counter_a())  # 1
print(counter_a())  # 2
print(counter_a())  # 3
print(counter_b())  # 1 — окремий власний "count", а не спільний з counter_a

1
2
3
1


`increment` звертається до `count` — змінної з **навколишньої** функції `make_counter`, а не до своєї власної. Це і є замикання: внутрішня функція "замикає" в собі змінні зовнішньої, навіть коли зовнішня функція вже завершила роботу.

`nonlocal count` каже Python: "не створюй нову локальну `count` всередині `increment`, використовуй ту, що в `make_counter`". Без цього рядка `count += 1` впаде з `UnboundLocalError` — Python побачить присвоєння `count = ...` і вирішить, що це нова локальна змінна, а не та, що ззовні.

`counter_a` і `counter_b` — два незалежні виклики `make_counter()`, тому в кожного своя окрема `count`. Саме ця "пам'ять між викликами" знадобиться нам за мить у `wrapper`-і декоратора.

In [4]:
def broken_counter():
    count = 0

    def increment():
        count += 1  # немає nonlocal — Python вважає count локальною ще ДО цього рядка
        return count

    return increment


bad = broken_counter()
try:
    bad()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value


Навмисна помилка: без `nonlocal` рядок `count += 1` еквівалентний `count = count + 1` — а це присвоєння всередині функції автоматично робить `count` **локальною** змінною для всього тіла `increment`, включно з правою частиною виразу. Python намагається прочитати ще не створену локальну `count` і падає. Це одна з найчастіших пасток при першому знайомстві з замиканнями — тепер ви бачили її "наосліп", а не лише в описі.

## CREATE — від болю до рішення

### Етап 1. Наївна реалізація

Три ролі: `guest` (тільки читає), `user` (звичайний користувач), `admin` (може все). Шість дій блогу — кожна перевіряє роль "в лоб", через `if`.

In [5]:
# Поточний користувач — змінюємо для тестування
current_user = {}

In [6]:
def view_post(post_id):
    if current_user["role"] not in ["guest", "user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


def create_post(title):
    if current_user["role"] not in ["user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"✍️  {current_user['name']} створює пост: '{title}'")


def edit_post(post_id):
    if current_user["role"] not in ["user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


def publish_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


def delete_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


def archive_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"📁  {current_user['name']} архівує пост #{post_id}")

In [7]:
current_user = {"name": "Іван", "role": "guest"}
view_post(1)
create_post("Мій перший пост")   # заборонено гостю
delete_post(1)                    # заборонено гостю

print()
current_user = {"name": "Оля", "role": "admin"}
view_post(1)
create_post("Важливе оголошення")
delete_post(1)

👁️  Іван переглядає пост #1
❌ Доступ заборонено!
❌ Доступ заборонено!

👁️  Оля переглядає пост #1
✍️  Оля створює пост: 'Важливе оголошення'
🗑️  Оля видаляє пост #1


### Етап 2. Де тут біль

Порахуйте, скільки разів у коді вище повторюється:

```python
if current_user["role"] not in [...]:
    print("❌ Доступ заборонено!")
    return
```

**Шість разів** — в усіх шести функціях. Тепер уявіть: що якщо функцій не 6, а 60? Що якщо потрібно змінити текст повідомлення про помилку — доведеться лізти в кожну з них.

In [8]:
# Менеджер: "Додайте роль moderator — може редагувати й публікувати, але не видаляти"
# Доводиться вручну заходити в КОЖНУ функцію:

def edit_post_v2(post_id):
    if current_user["role"] not in ["user", "admin", "moderator"]:  # ← додали тут
        print("❌ Доступ заборонено!")
        return
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


def publish_post_v2(post_id):
    if current_user["role"] not in ["admin", "moderator"]:  # ← і тут
        print("❌ Доступ заборонено!")
        return
    print(f"📢  {current_user['name']} публікує пост #{post_id}")

print("Дві функції змінено заради ОДНІЄЇ нової ролі — і так ще в чотирьох місцях попереду 🤦")

Дві функції змінено заради ОДНІЄЇ нової ролі — і так ще в чотирьох місцях попереду 🤦


Логіка доступу **розкидана** по шести місцях замість того, щоб жити в одному. Кожна зміна бізнес-правила — це похід по всьому файлу. Це не масштабується.

### Етап 3. Ідея рішення

Що якщо "загорнути" будь-яку функцію в перевірку доступу — написати одну функцію-обгортку, яка сама вирішує, викликати оригінал чи ні? Саме для цього щойно знадобились дві властивості функцій із CONCEPT: передати функцію як аргумент і повернути нову функцію.

In [9]:
def require_admin(func):
    '''Обгортка: перевіряє роль перед виконанням'''

    def wrapper():
        if current_user["role"] != "admin":
            print("❌ Доступ заборонено! Потрібна роль: admin")
            return
        func()  # роль підходить — викликаємо оригінал

    return wrapper


def delete_everything():
    print("🗑️ Видалено все!")


safe_delete = require_admin(delete_everything)  # "загортаємо" функцію

current_user = {"name": "Гість", "role": "guest"}
safe_delete()  # заблоковано

current_user = {"name": "Адмін", "role": "admin"}
safe_delete()  # дозволено

❌ Доступ заборонено! Потрібна роль: admin
🗑️ Видалено все!


Логіка перевірки доступу тепер **в одному місці** — всередині `require_admin`. `delete_everything` про роль нічого не знає — вона просто видаляє пости. `safe_delete` — нова функція, яка спершу перевіряє, а потім (якщо можна) викликає оригінал. Це і є декоратор, ще без спеціального синтаксису.

### Етап 4. Синтаксис `@`

Замість:

```python
safe_delete = require_admin(delete_everything)
```

Python дозволяє написати:

```python
@require_admin
def delete_everything():
    ...
```

Це **рівно те саме** — `@` лише скорочує запис "визнач функцію, а потім одразу оберни її в декоратор".

In [10]:
@require_admin
def rename_post(post_id, new_title):
    print(f"✏️  Перейменовано пост #{post_id} на «{new_title}»")


current_user = {"name": "Адмін", "role": "admin"}
try:
    rename_post(1, "Новий заголовок")
except TypeError as e:
    print("TypeError:", e)

TypeError: require_admin.<locals>.wrapper() takes 0 positional arguments but 2 were given


Навмисна помилка: `wrapper()` у `require_admin` оголошено **без параметрів**, тому виклик `func()` всередині теж без аргументів — а `rename_post` вимагає два. `require_admin` підходив лише для функцій без параметрів. Потрібен універсальний спосіб "передати все, що прийшло, далі".

In [11]:
def require_admin_v2(func):
    def wrapper(*args, **kwargs):          # приймаємо будь-які аргументи...
        if current_user["role"] != "admin":
            print("❌ Доступ заборонено! Потрібна роль: admin")
            return
        return func(*args, **kwargs)        # ...і передаємо їх далі, у оригінал
    return wrapper


@require_admin_v2
def rename_post(post_id, new_title):
    print(f"✏️  Перейменовано пост #{post_id} на «{new_title}»")


current_user = {"name": "Адмін", "role": "admin"}
rename_post(1, "Новий заголовок")  # тепер працює

✏️  Перейменовано пост #1 на «Новий заголовок»


`*args` збирає всі позиційні аргументи в кортеж, `**kwargs` — всі іменовані в словник. `func(*args, **kwargs)` розпаковує їх назад при виклику оригіналу. Тепер `wrapper` підходить для функції з **будь-якою** сигнатурою — нуль, один чи п'ять аргументів.

## CONCEPT — `functools.wraps`: декоратор ховає справжнє ім'я функції

Ще одна деталь, перш ніж повертатись до сценарію блогу.

In [12]:
def simple_decorator(func):
    def wrapper(*args, **kwargs):
        '''Docstring обгортки, а не оригіналу.'''
        return func(*args, **kwargs)
    return wrapper


@simple_decorator
def complex_math(x, y):
    '''Виконує складну математику.'''
    return x ** y


print(complex_math.__name__)  # мало бути 'complex_math'
print(complex_math.__doc__)   # мало бути 'Виконує складну математику.'

wrapper
Docstring обгортки, а не оригіналу.


Ззовні `complex_math` тепер виглядає як функція на ім'я `"wrapper"` без власного docstring — Python не бачить різниці між `wrapper` і будь-якою іншою функцією, тому `__name__`/`__doc__` беруться від того, що реально повернув декоратор. Це ламає дебаг, автогенеровану документацію, і будь-який код, що покладається на `func.__name__` (наприклад, логування).

In [13]:
import functools


def proper_decorator(func):
    @functools.wraps(func)   # копіює __name__, __doc__, __module__... з func у wrapper
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@proper_decorator
def complex_math(x, y):
    '''Виконує складну математику.'''
    return x ** y


print(complex_math.__name__)  # 'complex_math' ✅
print(complex_math.__doc__)   # 'Виконує складну математику.' ✅

complex_math
Виконує складну математику.


`@functools.wraps(func)` — теж декоратор, застосований до самого `wrapper`: він копіює метадані (`__name__`, `__doc__`, `__module__` тощо) з оригінальної функції на обгортку. Правило на майбутнє: **кожен production-декоратор повинен мати `@functools.wraps(func)`** на `wrapper`-і — саме так ми напишемо `require_role` нижче.

### Етап 5. Декоратор-фабрика: одна логіка, будь-яка роль

`require_admin_v2` жорстко перевіряє саме `"admin"`. Потрібен універсальний варіант, що приймає список дозволених ролей.

In [14]:
def require_role(*allowed_roles):
    '''Декоратор-фабрика: приймає ролі, повертає готовий декоратор.'''
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user["role"] not in allowed_roles:
                print(f"❌ Доступ заборонено! Потрібна роль: {' або '.join(allowed_roles)}")
                return
            return func(*args, **kwargs)
        return wrapper
    return decorator

Три рівні функцій, кожен зі своєю роботою:

| Рівень | Що робить |
|---|---|
| `require_role(*allowed_roles)` | Зовнішня — приймає список ролей, наприклад `"admin", "moderator"` |
| `decorator(func)` | Середня — приймає функцію, яку захищаємо |
| `wrapper(*args, **kwargs)` | Внутрішня — те, що реально виконується замість оригіналу |

`require_role("admin", "moderator")` спочатку повертає `decorator` (запам'ятавши ролі через замикання), і лише потім `decorator` застосовується до конкретної функції — саме тому це і фабрика декораторів, а не сам декоратор.

In [15]:
@require_role("guest", "user", "admin", "moderator")
def view_post(post_id):
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


@require_role("user", "admin")
def create_post(title):
    print(f"✍️  {current_user['name']} створює пост: '{title}'")


@require_role("user", "admin", "moderator")
def edit_post(post_id):
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


@require_role("admin", "moderator")
def publish_post(post_id):
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


@require_role("admin")
def delete_post(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


@require_role("admin")
def archive_post(post_id):
    print(f"📁  {current_user['name']} архівує пост #{post_id}")

print("Декоратори застосовано ✅")
print("view_post.__name__ =", view_post.__name__)  # завдяки functools.wraps — не 'wrapper'

Декоратори застосовано ✅
view_post.__name__ = view_post


In [16]:
current_user = {"name": "Гість Анонім", "role": "guest"}
print(f"=== {current_user['name']} ({current_user['role']}) ===")
view_post(42)
create_post("Мій пост")
delete_post(42)

=== Гість Анонім (guest) ===
👁️  Гість Анонім переглядає пост #42
❌ Доступ заборонено! Потрібна роль: user або admin
❌ Доступ заборонено! Потрібна роль: admin


In [17]:
current_user = {"name": "Марко-Модератор", "role": "moderator"}
print(f"=== {current_user['name']} ({current_user['role']}) ===")
view_post(42)
edit_post(42)
publish_post(42)
delete_post(42)  # модератору заборонено

=== Марко-Модератор (moderator) ===
👁️  Марко-Модератор переглядає пост #42
✏️  Марко-Модератор редагує пост #42
📢  Марко-Модератор публікує пост #42
❌ Доступ заборонено! Потрібна роль: admin


In [18]:
current_user = {"name": "Адмін Всесильний", "role": "admin"}
print(f"=== {current_user['name']} ({current_user['role']}) ===")
view_post(42)
create_post("Важливий анонс")
edit_post(42)
publish_post(42)
delete_post(42)
archive_post(42)

=== Адмін Всесильний (admin) ===
👁️  Адмін Всесильний переглядає пост #42
✍️  Адмін Всесильний створює пост: 'Важливий анонс'
✏️  Адмін Всесильний редагує пост #42
📢  Адмін Всесильний публікує пост #42
🗑️  Адмін Всесильний видаляє пост #42
📁  Адмін Всесильний архівує пост #42


### До і після: нова роль `superuser`

Менеджер повертається: «Додайте роль `superuser` — може видаляти й архівувати пости нарівні з `admin`».

**Завдання.** Допишіть ролі в `@require_role(...)` над двома функціями нижче. Тіла функцій не чіпайте. Функції повертають рядок-повідомлення, а заблокований виклик повертає `None` — так перевірки нижче бачать результат.

In [19]:
# YOUR CODE HERE — допишіть ролі так, щоб superuser мав ті самі права, що й admin
# BEGIN SOLUTION
@require_role("admin", "superuser")
def delete_post_v3(post_id):
    return f"{current_user['name']} видаляє пост #{post_id}"


@require_role("admin", "superuser")
def archive_post_v3(post_id):
    return f"{current_user['name']} архівує пост #{post_id}"
# END SOLUTION


current_user = {"name": "Супер Юзер", "role": "superuser"}
print(delete_post_v3(99))
print(archive_post_v3(99))
assert delete_post_v3(99) == "Супер Юзер видаляє пост #99"
assert archive_post_v3(99) == "Супер Юзер архівує пост #99"

current_user = {"name": "Звичайний user", "role": "user"}
assert delete_post_v3(99) is None      # user досі не може видаляти
print("OK — superuser додано однією зміною в рядку @")

🗑️  Супер Юзер видаляє пост #99


Порівняйте з Етапом 2, де нова роль означала лізти в тіло кожної функції. Тепер це одне слово в рядку `@require_role(...)` — тіла функцій не змінились.

### Етап 6. Кілька декораторів на одній функції — порядок має значення

Декоратори можна складати один на одному. Побудуємо простий `log_call`, який друкує факт виклику, і поєднаємо його з `require_role`.

In [20]:
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📞 виклик {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"↩️  {func.__name__} завершено")
        return result
    return wrapper


@log_call
@require_role("admin")
def moderate_comment_a(comment_id):
    print(f"🛡️  Модеровано коментар #{comment_id}")


@require_role("admin")
@log_call
def moderate_comment_b(comment_id):
    print(f"🛡️  Модеровано коментар #{comment_id}")


current_user = {"name": "Адмін", "role": "admin"}
print("--- log_call зовні (виконується першим) ---")
moderate_comment_a(1)

print()
print("--- require_role зовні (виконується першим) ---")
moderate_comment_b(1)

print()
current_user = {"name": "Гість", "role": "guest"}
print("--- гостю заборонено: log_call зовні все одно друкує спробу виклику ---")
moderate_comment_a(2)

--- log_call зовні (виконується першим) ---
📞 виклик moderate_comment_a(1,)
🛡️  Модеровано коментар #1
↩️  moderate_comment_a завершено

--- require_role зовні (виконується першим) ---
📞 виклик moderate_comment_b(1,)
🛡️  Модеровано коментар #1
↩️  moderate_comment_b завершено

--- гостю заборонено: log_call зовні все одно друкує спробу виклику ---
📞 виклик moderate_comment_a(2,)
❌ Доступ заборонено! Потрібна роль: admin
↩️  moderate_comment_a завершено


`@log_call` над `@require_role("admin")` читається знизу вгору при визначенні (`log_call(require_role("admin")(func))`), але **виконується згори вниз** при виклику: спочатку спрацьовує `log_call.wrapper`, і лише потім, зсередини нього, — `require_role.wrapper`. Тому в `moderate_comment_a` лог друкується навіть тоді, коли доступ зрештою заборонено (гість) — `log_call` встиг спрацювати ДО перевірки ролі. У `moderate_comment_b` порядок обернений: спершу перевіряється роль, і лише якщо доступ дозволено, спрацьовує лог. Той самий принцип лежить в основі middleware у веб-фреймворках (CORS → Auth → Validation → обробник) — просто там таких шарів-декораторів більше.

## CONCEPT — `functools.lru_cache`: готовий декоратор зі стандартної бібліотеки

Не кожен декоратор потрібно писати самому. `functools.lru_cache` кешує результати функції — не рахує двічі те, що вже рахував.

In [21]:
import time


def fib_slow(n):
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)


start = time.perf_counter()
result_slow = fib_slow(30)
time_slow = time.perf_counter() - start
print(f"fib_slow(30) = {result_slow}, час: {time_slow * 1000:.1f} мс")


@functools.lru_cache(maxsize=None)
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)


start = time.perf_counter()
result_fast = fib_fast(30)
time_fast = time.perf_counter() - start
print(f"fib_fast(30) = {result_fast}, час: {time_fast * 1000:.4f} мс")

print(f"Прискорення: ×{time_slow / time_fast:,.0f}")
print(fib_fast.cache_info())
assert result_slow == result_fast

fib_slow(30) = 832040, час: 85.6 мс
fib_fast(30) = 832040, час: 0.0437 мс
Прискорення: ×1,959
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


`fib_slow` рахує однакові підвипадки мільйони разів заново — рекурсивне дерево росте експоненційно. `@functools.lru_cache(maxsize=None)` запам'ятовує результат кожного унікального аргументу: другий виклик `fib_fast(10)` бере готову відповідь з кеша, а не рахує наново. Реальне прискорення видно у виводі вище — і воно росте ще сильніше з більшим `n`.

**Важливо:** `lru_cache` підходить лише для **чистих функцій** — коли однаковий вхід завжди дає однаковий вихід і функція не має побічних ефектів. Якщо результат залежить від чогось окрім аргументів (наприклад, від `current_user`, як у наших декораторах вище) — кешування дасть застарілу, неправильну відповідь.

## Не лише доступ: декоратор `@timer`

Декоратор — загальна ідея: будь-яку поведінку «до» або «після» виклику можна винести з функції. Ось декоратор, що вимірює час виконання будь-якої функції.

Порівняємо дві функції, які рахують одне й те саме: суму чисел від 0 до n−1. Перша проходить усі числа — `O(n)`, друга рахує формулою Гауса — `O(1)` (урок 8).

In [ ]:
import time


def timer(func):
    """Друкує, скільки часу виконувалась функція."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱️  {func.__name__} виконалась за {elapsed * 1000:.3f} мс")
        return result
    return wrapper


@timer
def sum_by_loop(n):
    """Сумує числа по одному — O(n)."""
    return sum(range(n))


@timer
def sum_by_formula(n):
    """Формула Гауса — O(1)."""
    return n * (n - 1) // 2


print(sum_by_loop(1_000_000))
print(sum_by_formula(1_000_000))
assert sum_by_loop(1000) == sum_by_formula(1000)
assert sum_by_loop.__name__ == "sum_by_loop"   # functools.wraps зберіг ім'я

Точні мілісекунди на кожному комп'ютері свої, але різниця завжди на порядки: цикл по мільйону чисел проти однієї формули. Зверни увагу: `timer` нічого не знає про те, що саме рахує функція. Він працює з будь-якою — як `require_role` працює з будь-якою функцією блогу.

## Підсумок

Декоратор — функція, яка приймає іншу функцію і повертає нову, "покращену" версію. Жодної магії:

```text
🔧 require_role("admin")
        ↓ загортає
📦 delete_post
        ↓
   нова функція: спочатку перевіряє роль, потім (якщо ок) викликає оригінал
```

**Коли тягнутися за декоратором:** коли один і той самий шматок коду повторюється на початку чи в кінці багатьох функцій — перевірка прав, логування, кешування, вимірювання часу, повторна спроба при помилці.

> Декоратор відповідає на питання **«ЯК запустити функцію»**, а не «що вона робить». `delete_post` не повинна знати про ролі чи логування — це не її робота.

```python
# Шаблон декоратора без параметрів
def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # до
        result = func(*args, **kwargs)
        # після
        return result
    return wrapper


# Шаблон декоратора-фабрики (з параметрами)
def my_decorator_with_args(param):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator
```

👉 Повний довідник — [Функції та функціональне програмування](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/reference/python_core/functions/) і [Простори імен / LEGB](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/reference/python_core/namespaces_legb/). Глибша архітектурна тема — стек декораторів як middleware, декоратори класів, `lru_cache` у продакшн-системах — чекає в М2.

## TRANSFER — самостійне завдання

Нова, структурно ідентична задача: замість «дозволити тільки перерахованим ролям» — навпаки, **заборонити** перерахованим ролям, усім іншим дозволити.

Напишіть декоратор-фабрику `deny_role(*blocked_roles)`, застосуйте її до функції `moderate_comment(comment_id)` (заборонити `"guest"`), і переконайтесь, що всі перевірки нижче проходять.

In [22]:
def deny_role(*blocked_roles):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user["role"] in blocked_roles:
                print(f"❌ Роль '{current_user['role']}' заблокована для цієї дії")
                return None
            return func(*args, **kwargs)
        return wrapper
    return decorator


# BEGIN SOLUTION
@deny_role("guest")
def moderate_comment(comment_id):
    return f"Коментар #{comment_id} модеровано"
# END SOLUTION


current_user = {"name": "Гість", "role": "guest"}
assert moderate_comment(1) is None

current_user = {"name": "Юзер", "role": "user"}
assert moderate_comment(2) == "Коментар #2 модеровано"

current_user = {"name": "Адмін", "role": "admin"}
assert moderate_comment(3) == "Коментар #3 модеровано"

current_user = {"name": "Модератор", "role": "moderator"}
assert moderate_comment(4) == "Коментар #4 модеровано"

# functools.wraps справді зберіг метадані
assert moderate_comment.__name__ == "moderate_comment"

# декоратор приймає й іменовані аргументи так само, як позиційні
@deny_role("guest")
def greet(name, greeting="Привіт"):
    return f"{greeting}, {name}!"

current_user = {"name": "Юзер", "role": "user"}
assert greet("Олена") == "Привіт, Олена!"
assert greet("Марко", greeting="Вітаю") == "Вітаю, Марко!"

current_user = {"name": "Гість", "role": "guest"}
assert greet("Хтось") is None

# декоратор можна перевикористати з іншим набором заблокованих ролей
@deny_role("guest", "user")
def admin_only_action():
    return "виконано"

current_user = {"name": "Модератор", "role": "moderator"}
assert admin_only_action() == "виконано"

current_user = {"name": "Юзер", "role": "user"}
assert admin_only_action() is None

print("Усі 10 перевірок пройдено ✅")

❌ Роль 'guest' заблокована для цієї дії
❌ Роль 'guest' заблокована для цієї дії
❌ Роль 'user' заблокована для цієї дії
Усі 10 перевірок пройдено ✅


## ✅ Самоперевірка (5 запитань)

**1.** У функції `make_counter` з розділу «Місток до декоратора» видалили рядок `nonlocal count`. Що станеться при виклику `increment()`?

<details><summary>Відповідь</summary><code>UnboundLocalError</code> — рядок <code>count += 1</code> без <code>nonlocal</code> трактується Python як створення нової <b>локальної</b> змінної <code>count</code> усередині <code>increment</code>, а читання значення відбувається до її створення (бо <code>+=</code> спершу читає поточне значення).</details>

**2.** Навіщо `wrapper` у декораторі приймає `*args, **kwargs`, а не просто `(post_id)`?

<details><summary>Відповідь</summary>Щоб один і той самий декоратор підходив для функцій з будь-якою кількістю та типом аргументів (наприклад, <code>view_post(post_id)</code> з одним аргументом і <code>create_post(title)</code> з іншим, або функція без аргументів) — <code>wrapper</code> не повинен знати наперед сигнатуру функції, яку загортає.</details>

**3.** Що станеться з `delete_post.__name__`, якщо прибрати `@functools.wraps(func)` з `decorator`?

<details><summary>Відповідь</summary>Стане <code>"wrapper"</code> замість <code>"delete_post"</code> — без <code>functools.wraps</code> задекорована функція втрачає ім'я та докстрінг оригіналу, підмінюючись іменем внутрішньої функції <code>wrapper</code>.</details>

**4.** Чим декоратор без параметрів (`def require_admin(func): ...`) відрізняється рівнями вкладеності від декоратора-фабрики з параметрами (`def require_role(*roles): ...`)?

<details><summary>Відповідь</summary>Декоратор без параметрів має два рівні: <code>require_admin(func)</code> одразу повертає <code>wrapper</code>. Фабрика має три рівні: <code>require_role(*roles)</code> повертає <code>decorator(func)</code>, яка вже повертає <code>wrapper</code> — зайвий рівень потрібен, щоб спершу "запам'ятати" параметри (<code>roles</code>), а вже потім отримати функцію, яку загортаємо.</details>

**5.** Логіку перевірки ролі можна було б викликати як звичайну функцію на початку кожного `def` (як у наївному варіанті на початку уроку). Коли декоратор — краще рішення, ніж такий виклик?

<details><summary>Відповідь</summary>Коли та сама поведінка ("до" або "після" виклику) потрібна багатьом функціям одразу, і хочеться, щоб вона була описана в одному місці, а не повторювалась у тілі кожної функції — це й називають cross-cutting concern (наскрізна турбота: доступ, логування, таймінг, кешування). Для одноразової перевірки всередині однієї конкретної функції звичайний виклик усередині тіла — цілком достатнє і простіше рішення.</details>

## Далі

Урок 10 («Ітератори й генератори») навчить функції віддавати результати по одному, не тримаючи всі дані в пам'яті. А глибший розгляд функцій як об'єктів першого класу — замикання в архітектурних патернах, стек декораторів як middleware, клас-декоратори — чекає в Модулі 2, позиція 18.